# Year 7 & Year 8 Extraction (Greater London)

- `year7_125activities.csv` — Year 7, activity columns limited to the 125
  suffixes stable across Year 1–Year 8, plus all non-activity columns.
- `year7_179activities.csv` — Year 7, activity columns limited to the 179
  suffixes stable across Year 3–Year 8, plus all non-activity columns.
- `year8_125activities.csv` — same as above, for Year 8.
- `year8_179activities.csv` — same as above, for Year 8.

**Key decisions:**

1. SPSS missing codes **-94 to -99** are recoded to `NaN`. This is applied to every
   extracted column except `serial` and the `wt_*` weight columns. No other
   missing-value handling is done — genuine `NaN`/system-missing values are left as is.
2. The Greater London filter is built from `LA_2023`'s value labels at runtime for
   each year (codes whose label starts with `E09`, excluding City of London)



In [1]:
import pyreadstat
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 50)

## 1. File paths

UKDA study numbers for reference: Year7=9136, Year8=9288.

In [25]:
YEAR_FILES = {
    7: r"C:\Users\Lenovo\Desktop\Dissertation\DATa\9136spss_2021-2022\UKDA-9136-spss\spss\spss28\active_lives_survey_nov_21-22_data_year_7_shared_20250103.sav",
    8: r"C:\Users\Lenovo\Desktop\Dissertation\DATa\9288spss_2022-2023\UKDA-9288-spss\spss\spss28\active_lives_survey_nov_22-23_data_year_8_shared_20250103.sav",
}

WHITELIST_125_PATH = r"C:\Users\Lenovo\Desktop\Dissertation\DATa\8_codebook\125_activities_composites_year1_to_year8.xlsx"
WHITELIST_179_PATH = r"C:\Users\Lenovo\Desktop\Dissertation\DATa\8_codebook\179_activities_composites_year3_to_year8.xlsx"

OUTPUT_DIR = r"C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Jingyi Hua\data\processed"


## 2. Variable definitions


In [ ]:
GEO_EXACT = ['LA_2023', 'Reg9', 'LondInOut']
DEMO_EXACT = ['Age16plus', 'Age9', 'Disab3']
DISTY_EXACT = [f'disty{i}_POP' for i in range(1, 14)]       
TOTAL_EXACT = ['MEMS7_ALL', 'MEMS7GR_ALL']
VOL_EXACT = ['VolAny', 'VolFrqB_Pop'] + [f'volint{i}_vol' for i in range(1, 8)]  
OTHER_EXACT = ['serial', 'mode', 'Number_Activities_150', 'month']
WEIGHT_PREFIX = 'wt_'

NON_ACTIVITY_EXACT_VARS = GEO_EXACT + DEMO_EXACT + DISTY_EXACT + TOTAL_EXACT + VOL_EXACT + OTHER_EXACT

ACTIVITY_PREFIXES = ['MEMS7_', 'MEMS7GR_', 'INOUTA_', 'INOUTB_', 'DAYS10P60GR_', 'MONTHS_12_']

MISSING_CODES = list(range(-99, -93))  # -99, -98, -97, -96, -95, -94

print(f"Non-activity exact-match variables: {len(NON_ACTIVITY_EXACT_VARS)}")

Non-activity exact-match variables: 33


## 3. Load activity whitelists from the comparison Excel files

In [13]:
df_125 = pd.read_excel(WHITELIST_125_PATH, sheet_name='1_Stable composites', header=3)
whitelist_125 = set(df_125['DV suffix'].dropna().astype(str).str.strip())

df_179 = pd.read_excel(WHITELIST_179_PATH, sheet_name='Stable composites')
whitelist_179 = set(df_179['DV suffix'].dropna().astype(str).str.strip())

print(f"125-suffix whitelist loaded: {len(whitelist_125)} suffixes")
print(f"179-suffix whitelist loaded: {len(whitelist_179)} suffixes")

assert len(whitelist_125) == 125, "125-whitelist count does not match expectation, check the sheet/column name"
assert len(whitelist_179) == 179, "179-whitelist count does not match expectation, check the sheet/column name"


125-suffix whitelist loaded: 125 suffixes
179-suffix whitelist loaded: 179 suffixes


## 4. Metadata helper

In [14]:
def read_metadata(filepath):
    """Read only the variable-level metadata of a .sav file (no data rows loaded)."""
    _, meta = pyreadstat.read_sav(filepath, metadataonly=True)
    return meta

## 5. Diagnostic check

In [15]:
missing_report = {}

for year, path in YEAR_FILES.items():
    meta = read_metadata(path)
    available = set(meta.column_names)
    missing_exact = [v for v in NON_ACTIVITY_EXACT_VARS if v not in available]
    weight_vars_found = sorted(v for v in available if v.startswith(WEIGHT_PREFIX))

    missing_report[year] = {
        'missing_exact_vars': missing_exact,
        'weight_vars_found': weight_vars_found,
    }

    print(f"Year {year}:")
    print(f"  Missing exact-match variables: {missing_exact if missing_exact else 'none'}")
    print(f"  Weight variables found ({len(weight_vars_found)}): {weight_vars_found}")
    print()

Year 7:
  Missing exact-match variables: none
  Weight variables found (8): ['wt_final', 'wt_final_AB', 'wt_final_AC', 'wt_final_B', 'wt_final_C', 'wt_final_online', 'wt_online_time', 'wt_time']

Year 8:
  Missing exact-match variables: none
  Weight variables found (8): ['wt_final', 'wt_final_AB', 'wt_final_AC', 'wt_final_B', 'wt_final_C', 'wt_final_online', 'wt_online_time', 'wt_time']



## 6. Geography filter

Builds the list of `LA_2023` numeric codes that belong to Greater London
(codes whose value label starts with `E09`), excluding City of London, directly
from each year's metadata. Also returns a code to (ONS_Code, Borough) lookup used
to add readable columns later.

In [16]:
def get_london_borough_codes(meta):
    """
    Return the list of LA_2023 codes belonging to Greater London (excluding City
    of London), plus a code -> (ONS_Code, Borough) lookup, from value-label metadata.
    """
    value_labels = meta.variable_value_labels.get('LA_2023', {})
    allowed_codes = []
    code_to_borough = {}

    for code, label in value_labels.items():
        label = str(label).strip()
        if label.startswith('E09') and 'city of london' not in label.lower():
            code_f = float(code)
            ons_code, _, borough_name = label.partition(' ')
            allowed_codes.append(code_f)
            code_to_borough[code_f] = (ons_code, borough_name)

    return allowed_codes, code_to_borough

## 7. Build the column list to read for one year

In [17]:
def build_columns_to_extract(available_columns):
    matched_non_activity = [v for v in NON_ACTIVITY_EXACT_VARS if v in available_columns]
    matched_non_activity += sorted(v for v in available_columns if v.startswith(WEIGHT_PREFIX))

    union_whitelist = whitelist_125 | whitelist_179
    matched_activity = []
    for prefix in ACTIVITY_PREFIXES:
        for col in available_columns:
            if col.startswith(prefix):
                suffix = col[len(prefix):]
                if suffix in union_whitelist:
                    matched_activity.append(col)

    return matched_non_activity, matched_activity

## 8. Missing value recoding

SPSS codes -94 to -99 are treated as missing and recoded to `NaN`. Applied to
every extracted column except `serial` and the `wt_*` weight columns.

In [18]:
def recode_missing(df, exclude_cols):
    cols_to_recode = [c for c in df.columns if c not in exclude_cols]
    df[cols_to_recode] = df[cols_to_recode].replace(MISSING_CODES, np.nan)
    return df

## 9. Per-year extraction

In [ ]:
def extract_one_year(year, filepath):
    print(f"--- Extracting Year {year} ---")
    meta = read_metadata(filepath)
    available_columns = set(meta.column_names)

    matched_non_activity, matched_activity = build_columns_to_extract(available_columns)
    columns_to_read = matched_non_activity + matched_activity

    df, full_meta = pyreadstat.read_sav(
        filepath,
        usecols=columns_to_read,
        apply_value_formats=False
    )

    allowed_codes, code_to_borough = get_london_borough_codes(full_meta)
    df = df[df['LA_2023'].isin(allowed_codes)].copy()
    df['serial'] = df['serial'].astype('Int64')

    exclude_from_recode = ['serial'] + [c for c in df.columns if c.startswith(WEIGHT_PREFIX)]
    df = recode_missing(df, exclude_from_recode)

    df['year'] = year

    value_label_rows = []
    for var in columns_to_read:
        var_label = full_meta.column_names_to_labels.get(var, '')
        val_labels = full_meta.variable_value_labels.get(var, {})
        if val_labels:
            for code, code_label in val_labels.items():
                value_label_rows.append([var, var_label, code, code_label])
        else:
            value_label_rows.append([var, var_label, None, None])

    value_label_table = pd.DataFrame(
        value_label_rows,
        columns=['Variable', 'VariableLabel', 'Code', 'CodeLabel']
    )
    value_label_table['year'] = year

    print(f"  Columns extracted: {len(columns_to_read)}, rows after London filter: {len(df)}")
    return df, value_label_table

## 10. Run extraction for Year 7 and Year 8

In [28]:
yearly_data = {}
yearly_value_labels = {}

for year, path in YEAR_FILES.items():
    df_year, vl_year = extract_one_year(year, path)
    yearly_data[year] = df_year
    yearly_value_labels[year] = vl_year

--- Extracting Year 7 ---


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_4824\3904965546.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['ONS_Code'] = df['LA_2023'].map(lambda c: code_to_borough.get(c, (None, None))[0])
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_4824\3904965546.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['Borough'] = df['LA_2023'].map(lambda c: code_to_borough.get(c, (None, None))[1])
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_4824\3904965546.py:24: PerformanceWarning: DataFrame is highly fragmented.  This is usuall

  Columns extracted: 1059, rows after London filter: 16139
--- Extracting Year 8 ---
  Columns extracted: 1059, rows after London filter: 16515


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_4824\3904965546.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['ONS_Code'] = df['LA_2023'].map(lambda c: code_to_borough.get(c, (None, None))[0])
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_4824\3904965546.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['Borough'] = df['LA_2023'].map(lambda c: code_to_borough.get(c, (None, None))[1])
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_4824\3904965546.py:24: PerformanceWarning: DataFrame is highly fragmented.  This is usuall

## 11. Split each year into a 125-whitelist version and a 179-whitelist version

In [ ]:
def select_activity_columns(df, whitelist_suffixes):
    keep_cols = []
    
    for col in df.columns:
        if col in NON_ACTIVITY_EXACT_VARS or col.startswith(WEIGHT_PREFIX):
            keep_cols.append(col)
            continue
        is_activity = False
        for prefix in ACTIVITY_PREFIXES:
            if col.startswith(prefix):
                is_activity = True
                suffix = col[len(prefix):]
                if suffix in whitelist_suffixes:
                    keep_cols.append(col)
                break
        if not is_activity:
            keep_cols.append(col)
    return df[keep_cols]

In [30]:
year7_125 = select_activity_columns(yearly_data[7], whitelist_125)
year7_179 = select_activity_columns(yearly_data[7], whitelist_179)
year8_125 = select_activity_columns(yearly_data[8], whitelist_125)
year8_179 = select_activity_columns(yearly_data[8], whitelist_179)

print("year7_125:", year7_125.shape)
print("year7_179:", year7_179.shape)
print("year8_125:", year8_125.shape)
print("year8_179:", year8_179.shape)

year7_125: (16139, 758)
year7_179: (16139, 1060)
year8_125: (16515, 758)
year8_179: (16515, 1060)


## 12. Combine the variable/value-label lookup table

In [31]:
combined_value_labels = pd.concat(yearly_value_labels.values(), ignore_index=True)
print("Value-label lookup table:", combined_value_labels.shape)
combined_value_labels.head()

Value-label lookup table: (17553, 5)


,Variable,VariableLabel,Code,CodeLabel,year
0,LA_2023,LA 2023 version,1.0,E07000223 Adur,7
1,LA_2023,LA 2023 version,3.0,E07000032 Amber Valley,7
2,LA_2023,LA 2023 version,4.0,E07000224 Arun,7
3,LA_2023,LA 2023 version,5.0,E07000170 Ashfield,7
4,LA_2023,LA 2023 version,6.0,E07000105 Ashford,7


## 13. Save outputs

In [32]:
year7_125.to_csv(f"{OUTPUT_DIR}\\year7_125activities.csv", index=False)
year7_179.to_csv(f"{OUTPUT_DIR}\\year7_179activities.csv", index=False)
year8_125.to_csv(f"{OUTPUT_DIR}\\year8_125activities.csv", index=False)
year8_179.to_csv(f"{OUTPUT_DIR}\\year8_179activities.csv", index=False)
combined_value_labels.to_csv(f"{OUTPUT_DIR}\\variable_value_labels_lookup_year7_8.csv", index=False)

print("Saved:")
print(f"  year7_125activities.csv: {year7_125.shape}")
print(f"  year7_179activities.csv: {year7_179.shape}")
print(f"  year8_125activities.csv: {year8_125.shape}")
print(f"  year8_179activities.csv: {year8_179.shape}")
print(f"  variable_value_labels_lookup_year7_8.csv: {combined_value_labels.shape}")

Saved:
  year7_125activities.csv: (16139, 758)
  year7_179activities.csv: (16139, 1060)
  year8_125activities.csv: (16515, 758)
  year8_179activities.csv: (16515, 1060)
  variable_value_labels_lookup_year7_8.csv: (17553, 5)
